# 🔧 Feature Engineering
## Marketing Intelligence AI Platform

**Purpose:** Transform raw campaign data into a rich feature set suitable for all four ML modules.

**Sections:**
1. Environment Setup
2. Load Cleaned Dataset
3. Core Metric Features (CTR, CPC, ROAS)
4. Date & Seasonality Features
5. Rolling Window Aggregations
6. Lag Features
7. Categorical Encoding
8. Feature Scaling
9. Train / Validation / Test Split
10. Feature Importance Preview (Random Forest proxy)
11. Save Feature Store & Preprocessor Artefacts

---
## 1. Environment Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
import joblib

from shared.data_loader import DataLoader
from shared.feature_engineering import FeatureEngineer
from shared.preprocess import SharedPreprocessor
from shared.constants import (
    DATE_COL, REVENUE_COL, SPEND_COL, CLICKS_COL,
    IMPRESSIONS_COL, CONVERSIONS_COL, CHANNEL_COL,
    CTR_COL, CPC_COL, ROAS_COL, FEATURE_STORE
)

pd.set_option('display.max_columns', 60)
sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

print('✅ Environment ready.')

---
## 2. Load Cleaned Dataset

In [ ]:
# TODO: Replace synthetic data with: df = DataLoader.load_cleaned()

np.random.seed(42)
N = 2000
dates = pd.date_range('2023-01-01', periods=N, freq='D')

df = pd.DataFrame({
    DATE_COL:         np.tile(dates[:500], 4)[:N],
    IMPRESSIONS_COL:  np.random.randint(500, 100000, N),
    CLICKS_COL:       np.random.randint(5, 5000, N),
    SPEND_COL:        np.random.uniform(10, 5000, N).round(2),
    CONVERSIONS_COL:  np.random.randint(0, 500, N),
    REVENUE_COL:      np.random.uniform(0, 25000, N).round(2),
    CHANNEL_COL:      np.random.choice(['google_ads','meta_ads','microsoft_ads'], N),
})

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
print(f'Cleaned dataset shape: {df.shape}')
df.head()

---
## 3. Core Metric Features (CTR, CPC, ROAS)

In [ ]:
# Click-Through Rate
df[CTR_COL] = (df[CLICKS_COL] / df[IMPRESSIONS_COL]).replace([np.inf, -np.inf], np.nan)

# Cost Per Click
df[CPC_COL] = (df[SPEND_COL] / df[CLICKS_COL]).replace([np.inf, -np.inf], np.nan)

# Return on Ad Spend
df[ROAS_COL] = (df[REVENUE_COL] / df[SPEND_COL]).replace([np.inf, -np.inf], np.nan)

# Cost Per Acquisition
df['cpa'] = (df[SPEND_COL] / df[CONVERSIONS_COL].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

# Revenue Per Click
df['rpc'] = (df[REVENUE_COL] / df[CLICKS_COL].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

print('Core metric features added:', [CTR_COL, CPC_COL, ROAS_COL, 'cpa', 'rpc'])
df[[CTR_COL, CPC_COL, ROAS_COL, 'cpa', 'rpc']].describe()

---
## 4. Date & Seasonality Features

In [ ]:
df['year']         = df[DATE_COL].dt.year
df['month']        = df[DATE_COL].dt.month
df['day']          = df[DATE_COL].dt.day
df['day_of_week']  = df[DATE_COL].dt.dayofweek        # 0=Mon, 6=Sun
df['week_of_year'] = df[DATE_COL].dt.isocalendar().week.astype(int)
df['quarter']      = df[DATE_COL].dt.quarter
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

# TODO: Add public holiday flags using a holiday library.

date_features = ['year','month','day','day_of_week','week_of_year','quarter','is_weekend']
print('Date features added:', date_features)
df[date_features].head()

---
## 5. Rolling Window Aggregations

In [ ]:
df = df.sort_values(DATE_COL).reset_index(drop=True)

for window in [7, 14, 30]:
    for col in [REVENUE_COL, SPEND_COL, ROAS_COL, CTR_COL]:
        df[f'{col}_roll{window}_mean'] = df[col].rolling(window, min_periods=1).mean()
        df[f'{col}_roll{window}_std']  = df[col].rolling(window, min_periods=1).std()

rolling_cols = [c for c in df.columns if 'roll' in c]
print(f'{len(rolling_cols)} rolling features added.')
df[rolling_cols[:6]].head()

---
## 6. Lag Features

In [ ]:
for lag in [1, 7, 14]:
    for col in [REVENUE_COL, SPEND_COL, ROAS_COL]:
        df[f'{col}_lag{lag}'] = df[col].shift(lag)

lag_cols = [c for c in df.columns if 'lag' in c]
print(f'{len(lag_cols)} lag features added.')

# Drop rows with NaN lags introduced by shifting
df = df.dropna(subset=lag_cols).reset_index(drop=True)
print(f'Shape after dropping lag NaN rows: {df.shape}')

---
## 7. Categorical Encoding

In [ ]:
# Label encode the channel column
le = LabelEncoder()
df['channel_encoded'] = le.fit_transform(df[CHANNEL_COL])

print('Channel label mapping:')
for idx, cls in enumerate(le.classes_):
    print(f'  {idx} → {cls}')

# TODO: Add one-hot encoding for high-cardinality categoricals.
# TODO: Consider TargetEncoder for campaign_id.
joblib.dump(le, '../models/preprocessors/encoder.pkl')
print('\n✅ Encoder saved to models/preprocessors/encoder.pkl')

---
## 8. Feature Scaling

In [ ]:
feature_cols = [
    c for c in df.columns
    if c not in [DATE_COL, CHANNEL_COL, REVENUE_COL]
    and df[c].dtype in [np.float64, np.int64]
]

print(f'Feature columns ({len(feature_cols)}):')
print(feature_cols)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols].fillna(0))
X_df = pd.DataFrame(X_scaled, columns=feature_cols)

joblib.dump(scaler, '../models/preprocessors/scaler.pkl')
joblib.dump(feature_cols, '../models/preprocessors/feature_columns.pkl')
print('\n✅ Scaler and feature columns saved to models/preprocessors/')

---
## 9. Train / Validation / Test Split

In [ ]:
y = df[REVENUE_COL].values
X = X_df.values

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.10, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.111, random_state=42)

print(f'Train      : {X_train.shape[0]:>6,} rows ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validation : {X_val.shape[0]:>6,} rows ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test       : {X_test.shape[0]:>6,} rows ({X_test.shape[0]/len(X)*100:.1f}%)')

# TODO: Use time-based split for time-series data instead of random split.

---
## 10. Feature Importance Preview (Random Forest Proxy)

In [ ]:
# Quick Random Forest fit to rank features before running the full model pipeline
rf = RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'feature':    feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig = px.bar(
    importance_df, x='importance', y='feature', orientation='h',
    title='Top-20 Feature Importances (Random Forest proxy)',
    color='importance', color_continuous_scale='Blues',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

---
## 11. Save Feature Store & Preprocessor Artefacts

In [ ]:
# Save feature store
df.to_csv(f'../{FEATURE_STORE}', index=False)
print(f'✅ Feature store saved → {FEATURE_STORE}')
print(f'   Shape: {df.shape}')
print(f'   Columns ({len(df.columns)}): {list(df.columns)}')

# TODO: Save train/val/test splits to data/processed/
# pd.DataFrame(X_train, columns=feature_cols).to_csv('../data/processed/train.csv', index=False)
# pd.DataFrame(X_val,   columns=feature_cols).to_csv('../data/processed/validation.csv', index=False)
# pd.DataFrame(X_test,  columns=feature_cols).to_csv('../data/processed/test.csv', index=False)